# Performance.py

This notebook is used to calculate the performance of a `sentence-transformers` model on a predefined set of validation examples. Performance is measured in several dimensions, each broken down by the number of neighbors `K` retrieved by the ANN search:

* Top-K accuracy: the percentage of the time that the correct standardized code is in the K highest scoring search results
* Mean rank: the average position (1st, 2nd, 3rd, etc.) in the list of returned neighbors (sorted by score) of the correct standardized code, when present
* Mean encoding time: the time to process a nonstandard input through the LLM and convert it to a vector embedding that can be searched against
* Mean search time: the time it takes to retrieve the list of neighbors
* Mean cosine similarity: the average cosine similarity between the nonstandard input and the **highest** scoring search result--this result is not guaranteed to be the correct answer

For the purposes of this script, vector databases (e.g. AI Search) are _not_ used; the goal of this script is assessing accuracy and relevance metrics, rather than perfectly measuring search speed, so encoded LOINC vectors are retained in memory.


## Setup

In the package installation below, we've had some version mismatching with `mlflow`, `skinnyflow`, and a module within `sentence-transformers`. The `--no-cache-dir` flag forces a fresh fetch so that stable, rebuilt versions are used each time, which mitigates this issue. If issues with either of the `flow` packages appear thereafter, they can be safely ignored.

In [ ]:
pip install --no-cache-dir azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec sentence-transformers hnswlib

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(
    vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential
)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = "workspaceblobstore"

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [ ]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Remainder of imports and some global constants we'll be using

In [ ]:
import random
import time
from typing import List

# MODEL VARIABLES
MODEL_NAME = "intfloat/e5-base-v2"
EMBEDDING_SIZE = 768

# The name of the file in blob storage of the model's pickled vectors
EMBEDDING_FILE = "loinc_lab_names_intfloat_e5-base-v2_20251007"

# VALIDATION VARIABLES
VALIDATION_FILE = "./validation_set_positive_pairs.txt"
K_VALUES = [1, 3, 5, 10]

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), while embedding tensor files and any HNSW `.index` files do not, and can simply be loaded directly from storage.

In [ ]:
# Load up the validation set data
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

## Step 2: Unpickle Embeddings

Using our mounted file system, we can directly open the embedding file and unpickle it. Remember, each embedding file is stored as a dictionary of not just the embeddings computed by the `sentence-transformers` model, but the standard LOINC codes associated with those embeddings. These are important later for measuring accuracy.

In [ ]:
import pickle

# Load up the pre-computed embeddings from the datastore
with fs.open(EMBEDDING_FILE) as fp:
    cache_data = pickle.load(fp)

name_codes = cache_data["codes"]
embeddings = cache_data["embeddings"]

# We move the embedded vectors to CPU for optimized searching later, and
# because the compute instance attached to this notebook is unlikely to
# have GPU capabilities, so this avoids errors
embeddings = embeddings.cpu().numpy()

## Step 3: Load HNSW Index

Whether computed from a previous Azure run, or computed locally and uploaded, we will use the embeddings to populate an HNSW index for fast Approximate Nearest Neighbor searching. The parameter values below govern the depth / connectivity of the search, but note that if the index was previously constructed, only the `EF_SEARCH` value will impact performance.

The `hnswlib` package _cannot_ directly open Azure binary files, which is how the FileSystemMount accesses and passes them. So what we need to do instead is first copy the file from the remote mount to local, working memory, and then we can access and open it. Once we've done that, it should be locally persisted for the remainder of our session.

**Important:** This cell will leave a copy of the `.index` file in local, working memory, which should be deleted when you are finished with your computing session. Look in the sidebar to the left, named `Notebooks`, and check under the folder name containing this notebook (e.g. `Users/brandon.mader/`). Right click on the `.index` file and simply delete it, which will free up resources, reduce costs, and improve other compute that might be relying on the compute instance.


In [ ]:
import hnswlib

# ANN INDEX VARIABLES
EF_CONSTRUCTION = 200
M_VALUE = 64
EF_SEARCH = 100

# The name of the HNSW index file for this particular model
INDEX_FP = "hnswlib_index_intfloat_e5-base-v2.index"

# Load up or create an index over the embedding data
index = hnswlib.Index(space="cosine", dim=EMBEDDING_SIZE)

# Azure will check blob storage first using the file mount
print("Checking for cached ANN index...")
if fs.exists(INDEX_FP):
    print("  Found cached index. Loading it...")

    # First, try to regularly load the index, in case we copied it here
    # from a previous run
    try:
        index.load_index(INDEX_FP)

    # If we can't open the file (because it's AzureML binary), then we
    # can create a local ported copy and open from that
    except:  # noqa
        try:
            fs.get(INDEX_FP, ".")
            index.load_index(INDEX_FP)

        # If that doesn't work then the file is beyond the reach of mortal
        # hands and is best left undisturbed, like all sleeping gods
        except:  # noqa
            print("Could not copy or load index")

else:
    print("No locally cached index found. Creating hierarchical index...")
    index.init_index(max_elements=len(embeddings), ef_construction=EF_CONSTRUCTION, M=M_VALUE)
    index.add_items(embeddings, list(range(len(embeddings))))

    # Default is to save to local, working memory, so we'll need to remote copy
    # to Azure blob storage just like the reverse of copying from blob storage
    index.save_index(INDEX_FP)
    fs.put("./" + INDEX_FP, INDEX_FP)

# The index should be holding approximately 276k embeddings so it better exceed 0
assert index.get_current_count() > 0
index.set_ef(EF_SEARCH)

## Step 4: Load Validation Set

With our file system mount, loading the validation set and preparing it for evaluation is straightforward. No need to worry about local copying for this data, Azure's `fs.open()` can simply parse the binary into a string codec for us.

In [ ]:
print("Loading validation set...")
examples = []
with fs.open(VALIDATION_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            examples.append(line_str.strip().split("|"))

## Step 5: Define Evaluation

This function specifies how the trial run with the model and validation data will be scored. It's by and large a simple dictionary-based tracking function that accumulates some numbers into lists partitioned out by the K-value associated with the run. When we use the `hnswlib` API to perform ANN, we get a pretty nested structure of a pair of lists denoting the search results and the _distances_ of those results to the input query. The only nuance to this function is unpacking those lists, converting distances into scores (since we want to measure similarity), and pairing up the found neighbor result with the standard LOINC code it represents, using our earlier unpickled Corpus ID indices.

In [ ]:
from sentence_transformers import SentenceTransformer


def predict_and_evaluate_validation_set(
    model: SentenceTransformer,
    ann_index: hnswlib.Index,
    standard_loinc_names: List[str],
    examples: List[List[str]],
    k_vals: List[int],
) -> None:
    """
    Compute performance statistics for a given model on a given set of validation
    data. The data is expected to be a list of lists in which the first element
    of each pair is the trial nonstandard free-text input, and the second element
    is the standardized code that should be mapped to. Computed statistics include
    Top-K accuracy for the given value of K, mean cosine similarity of the highest
    scoring result, and mean time to encode an input and perform semantic search.

    :param model: The sentence transformer model to evaluate.
    :param ann_index: A pre-computed HNSW index file over the embeddings that
      we want to match nonstandard inputs to.
    :param standard_loinc_names: A list of strings representing the names of
      the LOINC codes embedded in the `vector_db`. Note that the order of
      strings in the list should match the order of embeddings in the DB.
    :param examples: A list of lists of strings representing the experimental
      examples to evaluate.
    :param k_vals: A list of integers indicating how many neighbors should be
      retrieved from the DB across a range of trials.
    :returns: None
    """
    encoding_times = []
    cosine_sims = {k: [] for k in k_vals}
    times = {k: [] for k in k_vals}
    ranks = {k: [] for k in k_vals}
    examples_with_correct_output_in_top_k = {k: 0.0 for k in k_vals}

    random.shuffle(examples)
    examples = examples[:NUM_EXAMPLES_TO_VALIDATE]

    for e in examples:
        correct_code = e[0].strip()
        nonstandard_in = e[1].strip()

        start = time.time()
        enc = model.encode(nonstandard_in)
        encoding_times.append(time.time() - start)

        for k in k_vals:
            start = time.time()
            embedding_ids, distances = ann_index.knn_query(enc, k=k)
            hits = [
                {"corpus_id": id, "score": 1 - dist}
                for id, dist in zip(embedding_ids[0], distances[0])
            ]
            hits = sorted(hits, key=lambda x: x["score"], reverse=True)

            times[k].append(time.time() - start)
            cosine_sims[k].append(hits[0]["score"])

            # Check if correct answer is in the returned search results
            correct_in_top_k = False
            for idx, h in enumerate(hits):
                mapped_sentence = standard_loinc_names[h["corpus_id"]]  # ty: ignore
                if mapped_sentence == correct_code:
                    correct_in_top_k = True
                    # Hits is a 0-indexed list, so translate the index to the nth
                    # element of the list
                    ranks[k].append(idx + 1)
                    break
            if correct_in_top_k:
                examples_with_correct_output_in_top_k[k] += 1.0

    mean_encoding_time = round(float(sum(encoding_times)) / float(len(encoding_times)), 3)
    print(f"  Mean Encoding Time: {mean_encoding_time} seconds")

    for k in k_vals:
        mean_cosine_sim = round(float(sum(cosine_sims[k])) / float(len(cosine_sims[k])), 3)
        mean_encoding_search_time = round(float(sum(times[k])) / float(len(times[k])), 3)
        top_k_accuracy = round(examples_with_correct_output_in_top_k[k] / float(len(examples)), 5)
        mean_rank = round(float(sum(ranks[k])) / float(len(ranks[k])), 3)

        print(f"  Trial: Value for Top-K at K = {k}")
        print(f"    Top-K Accuracy: {top_k_accuracy * 100.0}%")
        print(f"    Mean Rank of Correct Code (when present): {mean_rank}")
        print(f"    Top-K Accuracy: {top_k_accuracy * 100.0}%")
        print(f"    Mean Cosine Similarity: {mean_cosine_sim}")
        print(f"    Mean Search Time: {mean_encoding_search_time}")

## Step 6: Perform Analysis

Now, all that's left is to instantiate the formal `sentence-transformers` model (which we need to do so we can encode the validation inputs) and compute our metrics!

In [ ]:
print("Instantiating language model...")
model = SentenceTransformer(MODEL_NAME)

# IMPORTANT: Change this value to calculate stats using more or less
# examples drawn from the validation set.
NUM_EXAMPLES_TO_VALIDATE = 1000

print("Predicting and computing stats for validation set...")
predict_and_evaluate_validation_set(model, index, name_codes, examples, K_VALUES)